In [11]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

In [12]:
df = pd.read_csv('grocerydb.csv')
df.head(3)

,name,store,category,brand,FPro,FPro_class,price,price percal,package_weight,Protein,Total Fat,Carbohydrate,"Sugars, total","Fiber, total dietary",Sodium,Cholesterol
0,Stonyfield Organic Whole Milk Strawberry Beet ...,Target,baby-food,Stonyfield,0.815250,3.0,5.29,0.043984,396.8930,5.050505,3.030303,12.121212,9.090909,0.000000,0.080808,0.010101
1,Stonyfield Organic Whole Milk Pear Spinach Man...,Target,baby-food,Stonyfield,0.815250,3.0,5.29,0.043984,396.8930,5.050505,3.030303,12.121212,9.090909,0.000000,0.080808,0.010101
2,Once Upon a Farm Organic Mama Blueberry Fruit ...,Target,baby-food,Once Upon a Farm,0.583219,3.0,2.79,0.055973,90.7184,1.098901,0.549451,13.186813,7.692308,2.197802,0.010989,0.000000


In [13]:
means = df.pivot_table(index='FPro_class', values=df.columns[6:], aggfunc='mean')
means

,Carbohydrate,Cholesterol,"Fiber, total dietary",Protein,Sodium,"Sugars, total",Total Fat,package_weight,price,price percal
FPro_class,,,,,,,,,,
0.0,20.977507,0.025024,4.091072,9.042560,0.051543,3.896688,6.115350,587.378800,4.948620,0.171296
1.0,19.579185,0.015323,4.551813,2.595861,1.789549,6.646934,27.337418,439.532205,5.568323,0.021264
2.0,11.039735,0.087272,2.912297,17.901563,0.874021,3.980252,24.458313,471.024711,5.206017,0.011843
3.0,41.127635,0.026071,3.079521,7.702438,28.520709,14.338213,12.279792,440.412681,4.493045,0.011543


In [14]:
px.line(means, log_y=True)

In [15]:
category_order = df.groupby("category")["FPro"].mean().sort_values().index.tolist()
px.violin(df, x="category", y="FPro", points=False, category_orders={"category": category_order})

In [16]:
lens = df.assign(length=df['name'].str.len())
lens.groupby('FPro_class')['length'].describe()

,count,mean,std,min,25%,50%,75%,max
FPro_class,,,,,,,,
0.0,3313.0,38.279505,17.380590,4.0,25.00,36.0,49.0,183.0
1.0,616.0,41.449675,16.008049,5.0,30.75,39.0,48.0,124.0
2.0,2426.0,46.265458,17.522522,5.0,34.00,44.0,57.0,139.0
3.0,19895.0,48.991154,17.844601,6.0,36.00,47.0,60.0,198.0


In [17]:
fig = px.violin(
    df,
    x="store",
    y="FPro",
    box=True,
    points=False,
    title="Which retailer (Target, Walmart, WholeFoods) has to most processed food?",
    category_orders={"store": ["Target", "Walmart", "WholeFoods"]},
)

fig.update_layout(
    xaxis_title="Retailer",
    yaxis_title="Food Processing Score",
)

fig.show()

In [18]:
category_rename_map = {
    'culinary-ingredients': 'Culinary ingredients',
    'drink-shakes-other': 'Drinks: shakes, others',
    'drink-coffee': 'Coffee drinks',
    'spices-seasoning': 'Spices, seasoning',
    'soup-stew': 'Soups, stews',
    'muffins-bagels': 'Muffins, bagels',
    'sauce-all': 'Sauces',
    'breakfast': 'Breakfast',
    'snacks-bars': 'Snacks',
    'drink-juice': 'Juice drinks',
    'mac-cheese': 'Mac and cheese',
    'pastry-chocolate-candy': 'Pastry, chocolate, candy',
    'snacks-mixes-crackers': 'Snacks',
    'ice-cream-dessert': 'Ice cream, desserts',
    'pudding-jello': 'Pudding, jello',
    'cookies-biscuit': 'Cookies, biscuits',
    'bread': 'Bread',
    'cakes': 'Cakes',
    'sausage-bacon': 'Sausages, bacon',
    'pizza': 'Pizza',
    'rolls-buns-wraps': 'Rolls, buns, wraps',
    'meat-packaged': 'Packaged meat',
    'snacks-chips': 'Snacks',
    'dairy-yogurt-drink': 'Dairy: yogurt, drink',
    'prepared-meals-dishes': 'Prepared meals',
    'seafood': 'Seafood',
    'snacks-dips-salsa': 'Snacks',
    'dressings': 'Dressings',
    'cereal': 'Breakfast',
    'cheese': 'Cheese',
    'milk-milk-substitute': 'Milk, milk substitute',
    'snacks-nuts-seeds': 'Snacks'
}

custom_cats = df.copy()
custom_cats["category"] = df[df["category"].isin(category_rename_map.keys())]["category"].replace(category_rename_map)

In [19]:
spacing = 1.5

# Sort categories from most to least processed
category_order = custom_cats.groupby("category")["FPro"].mean().sort_values(ascending=False).index.tolist()
x_positions = {cat: i * spacing for i, cat in enumerate(category_order)}
x_vals = list(x_positions.values())

# Normalize for color gradient — reversed for most processed = darkest
norm_x = np.array(x_vals[::-1]) / max(x_vals)  # reversed for color flip
colorscale = px.colors.sequential.YlOrRd
color_map = [px.colors.sample_colorscale(colorscale, [v])[0] for v in norm_x]

fig = go.Figure()

for i, cat in enumerate(category_order):
    cat_data = custom_cats.loc[custom_cats["category"] == cat, "FPro"]

    fig.add_trace(
        go.Violin(
            y=cat_data,
            x=[x_positions[cat]] * len(cat_data),
            name=cat,
            points=False,
            meanline_visible=True,
            width=spacing * 0.8,
            line_color="black",
            fillcolor=color_map[i],
            showlegend=False,
            spanmode="hard"
        )
    )

fig.update_layout(
    title={
        "text": "Which category of food is the most processed?",
        "x": 0.5,
        "xanchor": "center",
    },
    yaxis=dict(
        title="Food Processing Score",
        range=[-0.1, 1.1]
    ),
    xaxis=dict(
        tickvals=x_vals,
        ticktext=category_order,
        range=[min(x_vals) - spacing / 2, max(x_vals) + spacing / 2]
    ),
    width=spacing * len(category_order) * 40,
    height=600,
)

fig.show()

In [54]:
import plotly.graph_objects as go
import plotly.colors as pc

# Preprocessing
avg = (
    custom_cats[custom_cats["FPro_class"].isin([0, 3])]
    .groupby(["category", "FPro_class"])["price percal"]
    .mean()
    .unstack()
    .dropna()
    .rename(columns={0: "class_0", 3: "class_3"})
)
avg["max"] = avg[["class_0", "class_3"]].max(axis=1)
avg["overall_avg"] = custom_cats.groupby("category")["price percal"].mean()
avg = avg.sort_values("max", ascending=False).reset_index()

fig = go.Figure()

# Connecting lines
for cat, row in avg.iterrows():
    fig.add_trace(
        go.Scatter(
            x=[row["category"]] * 2,
            y=[row["class_0"], row["class_3"]],
            mode="lines",
            line=dict(color="lightgray", width=2),
            hoverinfo="skip",
            showlegend=False,
        )
    )

# Add traces with red for class 3 and green for class 0
for col, color, name in [
    ("class_0", pc.qualitative.Plotly[2], "Minimally Processed"),  # green
    ("class_3", pc.qualitative.Plotly[1], "Ultra Processed"),  # red
    # ("overall_avg", pc.qualitative.Plotly[0], "Overall Average"),
]:
    fig.add_trace(
        go.Scatter(
            x=avg["category"],
            y=avg[col],
            mode="markers",
            marker=dict(
                color=color,
                size=11 if "class" in col else 8,
                symbol="circle" if "class" in col else "diamond",
            ),
            name=name,
            hovertemplate=f"{name}: %{{y:.3f}}<extra></extra>",
        )
    )

bread_row = avg[avg["category"] == "Bread"].iloc[0]
x_bread = bread_row["category"]

# Get y-values for Bread class_0 and class_3
y_top = max(bread_row["class_0"], bread_row["class_3"])
y_bot = min(bread_row["class_0"], bread_row["class_3"])
y_middle = 10 ** ((np.log10(y_top) + np.log10(y_bot)) / 2)

# Adjust for left position of bracket
bracket_x = avg[avg["category"] == "Bread"].index[0] - 0.15  # Shift to the left

# Square bracket components (adjusted for left side)
fig.add_shape(
    type="line",
    x0=bracket_x - 0.15,
    x1=bracket_x - 0.15,
    y0=y_top,
    y1=y_middle * 1.1,
    xref="x",
    yref="y",
    line=dict(color="black", width=1),
)

fig.add_shape(
    type="line",
    x0=bracket_x - 0.15,
    x1=bracket_x - 0.15,
    y0=y_middle * 0.9,
    y1=y_bot,
    xref="x",
    yref="y",
    line=dict(color="black", width=1),
)

# Horizontal tops and bottoms of bracket (adjusted for left side)
fig.add_shape(
    type="line",
    x0=bracket_x - 0.15,
    x1=bracket_x + 0.05,
    y0=y_top,
    y1=y_top,
    xref="x",
    yref="y",
    line=dict(color="black", width=1),
)

fig.add_shape(
    type="line",
    x0=bracket_x - 0.15,
    x1=bracket_x + 0.05,
    y0=y_bot,
    y1=y_bot,
    xref="x",
    yref="y",
    line=dict(color="black", width=1),
)

fig.add_annotation(
    x=x_bread,
    y=0.545,
    xref="x",
    yref="paper",
    text="20x",
    showarrow=False,
    font=dict(size=13, color="black"),
    xshift=-15,
)

# Layout
fig.update_layout(
    title=dict(
        text="Want to Eat Clean? Prepare to Pay More<br><span style='font-size:14px; color:gray;'>Unprocessed Foods Cost Up to 20x More</span>",
        xanchor="center",
        x=0.5,
    ),
    xaxis=dict(title="Category", tickangle=-35, showgrid=False, tickfont=dict(size=11)),
    yaxis=dict(
        title="Avg Price per Calorie (Log Scale)",
        title_standoff=20,  # Adds space between title and axis
        type="log",
        tickvals=[0.005, 0.01, 0.02, 0.05, 0.1, 0.2],
        ticktext=["0.005", "0.01", "0.02", "0.05", "0.1", "0.2"],
        gridcolor="rgba(0,0,0,0.05)",
        zeroline=False,
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    width=1100,
    height=550,
    margin=dict(t=80, b=135, l=95, r=40),  # Increased left margin from 60 to 90
    legend=dict(
        yanchor="top",
        xanchor="right",
        y=0.98,
        x=1,
        bgcolor="rgba(255,255,255,0.95)",
        bordercolor="lightgray",
        borderwidth=1,
    ),
)

fig.show()